In [19]:
import pandas as pd
from currency_converter import CurrencyConverter
df = pd.read_csv("jobs.csv")

# Turning all salaries into RON
c = CurrencyConverter()
rates: dict[str, float] = {
	"RON": 1.0,
	"EUR": float(c.convert(1, "EUR", "RON")),
	"USD": float(c.convert(1, "USD", "RON")),
}
rate = df["currency"].map(rates)
df["salary_min_ron"] = df["salary_min"] * rate
df["salary_max_ron"] = df["salary_max"] * rate
df["salary_mid_ron"] = (df["salary_max_ron"] + df["salary_min_ron"]) / 2.00
# salaries with aberrant pays are flagged
df["salary_ok"] = df["salary_mid_ron"].between(999, 45000)


# turning the string dates of the jobs to actual dates
df["date_posted"] = pd.to_datetime(df["date_posted"], utc=True)
df["valid_through"] = pd.to_datetime(df["valid_through"], utc=True)

# Some jobs are listed like Timisoara (Timis), we split the city and county into 2 different collumns
parts = df["city"].str.split(" (", regex=False, expand=True)
df["city"] = parts[0].str.strip()
df["county"] = parts[2].fillna(parts[1]).str.rstrip(")")

df.head()

,title,company,city,salary_min,salary_max,currency,date_posted,url,employment_type,experience_months,education,industry,occupations,valid_through,description,salary_min_ron,salary_max_ron,salary_mid_ron,salary_ok,county
0,Account Manager,CONFIDENTIAL,Târgu Mureș,NaN,NaN,NaN,2026-09-25 12:33:48+00:00,https://www.ejobs.ro/user/locuri-de-munca/acco...,FULL_TIME,12.0,Absolvent,"Asigurări, Bănci / Servicii financiare, Farma,...","Achiziții, Farmacie, Medicină umană, MLM / Vân...",2026-10-24 21:00:00+00:00,Oportunitatea de dezvoltare profesională în ca...,NaN,NaN,NaN,False,NaN
1,Achizitor auto rulate,TIRIAC AUTO,Constanța,NaN,NaN,NaN,2026-09-17 16:50:20+00:00,https://www.ejobs.ro/user/locuri-de-munca/achi...,FULL_TIME,12.0,"Calificat, Absolvent","Mașini / Auto, Prestări servicii","Achiziții, Auto / Echipamente",2026-10-16 21:00:00+00:00,Ce <strong>responsabilități</strong> vei avea?...,NaN,NaN,NaN,False,NaN
2,📌Achizitor/Operator,Eurobusiness Tipar,București,NaN,NaN,NaN,2026-08-31 11:19:23+00:00,https://www.ejobs.ro/user/locuri-de-munca/achi...,FULL_TIME,36.0,Absolvent,Producție,Achiziții,2026-09-29 21:00:00+00:00,În rolul de Procurement Specialist / Specialis...,NaN,NaN,NaN,False,NaN
3,ACHIZITOR,SC Global Plast SRL,București,4000.0,6000.0,RON,2026-09-23 09:48:16+00:00,https://www.ejobs.ro/user/locuri-de-munca/achi...,FULL_TIME,36.0,"Calificat, Absolvent","Alimentară, Comerț / Retail, Farma, Transport ...","Achiziții, Administrativ / Logistică, Import -...",2026-10-22 21:00:00+00:00,- Asigurarea disponibilitatii stocurilor pentr...,4000.0,6000.0,5000.0,True,NaN
4,Agent vanzari,BN Business SRL,București,NaN,NaN,NaN,2026-09-15 11:34:07+00:00,https://www.ejobs.ro/user/locuri-de-munca/agen...,FULL_TIME,12.0,"Calificat, Absolvent",Comerț / Retail,"Achiziții, Auto / Echipamente, Merchandising /...",2026-10-14 21:00:00+00:00,Descrierea jobului:<br/>\n- Prezintă și promov...,NaN,NaN,NaN,False,NaN


In [20]:
df[["city", "county"]].value_counts().head(10)

city        county         
Iași        Iași               228
Bistrița    Bistrița-Năsăud     51
Satu Mare   Satu Mare           48
Galați      Galați              46
Târgoviște  Dâmbovița           28
Călărași    Călărași            24
Slatina     Olt                 23
Turda       Cluj                16
Sebeș       Alba                 7
Moreni      Dâmbovița            6
Name: count, dtype: int64

In [21]:
df[["salary_min_ron", "salary_max_ron", "salary_mid_ron"]].dropna().sample(10)

,salary_min_ron,salary_max_ron,salary_mid_ron
5367,9462.24,9462.24,9462.24
4996,6600.00,8000.00,7300.00
3975,4700.00,6300.00,5500.00
3471,2000.00,4000.00,3000.00
4512,3000.00,4000.00,3500.00
4676,5000.00,6000.00,5500.00
642,2500.00,2500.00,2500.00
1989,4000.00,4000.00,4000.00
4472,7300.00,7300.00,7300.00
2868,5000.00,6500.00,5750.00


In [22]:
df.sort_values("salary_mid_ron", ascending=False)[["title", "salary_min", "salary_max", "currency", "salary_mid_ron"]].head(10)

,title,salary_min,salary_max,currency,salary_mid_ron
4317,CAD Design Engineer,90000.0,100000.0,USD,432340.057138
3514,Medic specialist pentru Belgia,7000.0,24500.0,EUR,82794.600000
3535,MEDICI IN SUDUL FRANTEI,10000.0,20000.0,EUR,78852.000000
3092,National Sales Director - Enterprise Software ...,500.0,100000.0,RON,50250.000000
4309,INGINER PROIECTANT STRUCTURIST,500.0,99998.0,RON,50249.000000
3530,Medic stomatolog parodontolog - CLINICA Piatr...,7000.0,10000.0,EUR,44682.800000
991,INVESTMENT & ACQUISITIONS DIRECTOR,8000.0,8000.0,EUR,42054.400000
3482,Medic de Familie - Franta,6000.0,10000.0,EUR,42054.400000
2706,Broker Imobiliar | KLASS Estates & Finance,5000.0,10000.0,EUR,39426.000000
5315,Agent vanzari,4000.0,60000.0,RON,32000.000000


In [23]:
df["salary_ok"].value_counts()

salary_ok
False    4472
True     1995
Name: count, dtype: int64

In [24]:
from webscraping import fetch_urls_from_sitemap
urls = fetch_urls_from_sitemap("https://www.ejobs.ro/sitemap-listings-achizitii.xml")
print(len(urls))
print([u for u in urls if "content.ejobs.ro" in u])

index = fetch_urls_from_sitemap("https://www.ejobs.ro/sitemap-listings-index.xml")
print(len(index))

179
[]
68
